In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == "dualtest_experiments" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "DUALTEST"))

print(PROJECT_ROOT)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from experiment_utils import prepare_records
from target_model import HFLocalTarget
from reference_model import ReferenceModel
from scoring import score_texts
from generalization_sets import build_generalization_sets
from calibration import calibrate_1d, calibrate_2d
from sklearn.metrics import roc_auc_score

In [ ]:
TARGET_MODEL = "EleutherAI/pythia-12b"
REFERENCE_MODEL = "EleutherAI/pythia-410m"

TARGET_TAG = "pythia12b"
REFERENCE_TAG = "pythia410m"

DEVICE = "cuda"

In [ ]:
target = HFLocalTarget(
    model_name=TARGET_MODEL,
    device=DEVICE
)

reference = ReferenceModel(
    model_name=REFERENCE_MODEL,
    device=DEVICE
)

In [ ]:
def compute_auc_table(df):
    df = df.copy()

    df["esb_score"] = -np.log10(
        np.maximum(df["p_esb"], 1e-300)
    )

    return pd.DataFrame([
        {
            "metric": "run_length",
            "auc": roc_auc_score(df["label"], df["run_length"]),
        },
        {
            "metric": "edit_similarity",
            "auc": roc_auc_score(df["label"], df["edit_similarity"]),
        },
        {
            "metric": "esb_score",
            "auc": roc_auc_score(df["label"], df["esb_score"]),
        },
    ])

In [ ]:
def calibrate_dataset(df_normal, df_adv):
    member_scores = df_normal[df_normal["label"] == 1]["p_rlb"].tolist()
    nonmember_scores = df_normal[df_normal["label"] == 0]["p_rlb"].tolist()
    adversarial_scores = df_adv["p_rlb"].tolist()

    rlb_result = calibrate_1d(
        member_scores=member_scores,
        nonmember_scores=nonmember_scores,
        adversarial_scores=adversarial_scores,
        lower_is_suspicious=True,
    )

    member_esb = list(zip(
        df_normal[df_normal["label"] == 1]["edit_similarity"],
        df_normal[df_normal["label"] == 1]["p_esb"],
    ))

    nonmember_esb = list(zip(
        df_normal[df_normal["label"] == 0]["edit_similarity"],
        df_normal[df_normal["label"] == 0]["p_esb"],
    ))

    adversarial_esb = list(zip(
        df_adv["edit_similarity"],
        df_adv["p_esb"],
    ))

    esb_result = calibrate_2d(
        member=member_esb,
        nonmember=nonmember_esb,
        adversarial=adversarial_esb,
    )

    return rlb_result, esb_result

In [ ]:
def score_generalization_domain(domain, n_a=100, n_b=10):
    gen_sets = build_generalization_sets(
        n_a=n_a,
        n_b=n_b,
        domains=(domain,)
    )

    df_a = score_texts(
        texts=gen_sets[domain]["A"],
        target=target,
        reference=reference,
        dataset_name=f"generalization_{domain}_A",
        label=0,
    )

    df_a["set"] = "A"
    df_a["domain"] = domain

    df_b = score_texts(
        texts=gen_sets[domain]["B"],
        target=target,
        reference=reference,
        dataset_name=f"generalization_{domain}_B",
        label=0,
    )

    df_b["set"] = "B"
    df_b["domain"] = domain

    return pd.concat([df_a, df_b], ignore_index=True)

In [ ]:
records_book = prepare_records(
    dataset_name="booktection",
    n=1000,
    random_state=7,
    balance_labels=True,
)

texts = [r["text"] for r in records_book]
ids = [r["id"] for r in records_book]
labels = [r["label"] for r in records_book]
memberships = [r["estimated_membership"] for r in records_book]

df_book = score_texts(
    texts=texts,
    target=target,
    reference=reference,
    dataset_name="booktection",
    label=None,
    prefix_len=64,
    continuation_len=64,
    max_new_tokens=64,
)

df_book["id"] = ids
df_book["label"] = labels
df_book["membership"] = memberships
df_book["dataset"] = "booktection"

df_book.head()

In [ ]:
df_gen_book = score_generalization_domain(
    domain="book",
    n_a=100,
    n_b=10,
)

df_gen_book.head()

In [ ]:
out_dir = PROJECT_ROOT / "results"
out_dir.mkdir(exist_ok=True)

book_path = out_dir / f"booktection_{TARGET_TAG}_{REFERENCE_TAG}_n1000.csv"
gen_book_path = out_dir / f"generalization_book_{TARGET_TAG}_{REFERENCE_TAG}.csv"

df_book.to_csv(book_path, index=False)
df_gen_book.to_csv(gen_book_path, index=False)

print(book_path)
print(gen_book_path)

In [ ]:
book_auc = compute_auc_table(df_book)
book_rlb, book_esb = calibrate_dataset(df_book, df_gen_book)

print("AUC BookTection")
display(book_auc)

print("RLB:", book_rlb)
print("ESB:", book_esb)

In [ ]:
WIKIMIA_LENGTH = 256
PREFIX_LEN = WIKIMIA_LENGTH // 2
CONTINUATION_LEN = WIKIMIA_LENGTH // 2

meta = pd.read_csv(PROJECT_ROOT / "dataset/metadata/wikimia_metadata.csv")

meta_len = meta[
    meta["split"] == f"WikiMIA_length{WIKIMIA_LENGTH}"
].copy()

records_wiki = []

for _, row in meta_len.iterrows():
    path = PROJECT_ROOT / row["file_path"]
    text = path.read_text(encoding="utf-8").strip()

    records_wiki.append({
        "id": row["file_name"],
        "text": text,
        "label": row["label"],
        "membership": row["estimated_membership"],
        "dataset": f"wikimia_length{WIKIMIA_LENGTH}",
    })

len(records_wiki), pd.Series([r["label"] for r in records_wiki]).value_counts()

In [ ]:
texts = [r["text"] for r in records_wiki]
ids = [r["id"] for r in records_wiki]
labels = [r["label"] for r in records_wiki]
memberships = [r["membership"] for r in records_wiki]

df_wiki = score_texts(
    texts=texts,
    target=target,
    reference=reference,
    dataset_name=f"wikimia_length{WIKIMIA_LENGTH}",
    label=None,
    prefix_len=PREFIX_LEN,
    continuation_len=CONTINUATION_LEN,
    max_new_tokens=CONTINUATION_LEN,
)

df_wiki["id"] = ids
df_wiki["label"] = labels
df_wiki["membership"] = memberships
df_wiki["dataset"] = f"wikimia_length{WIKIMIA_LENGTH}"

df_wiki.head()

In [ ]:
df_gen_wiki = score_generalization_domain(
    domain="wiki",
    n_a=100,
    n_b=10,
)

df_gen_wiki.head()

In [ ]:
wiki_path = out_dir / f"wikimia_length{WIKIMIA_LENGTH}_{TARGET_TAG}_{REFERENCE_TAG}.csv"
gen_wiki_path = out_dir / f"generalization_wiki_{TARGET_TAG}_{REFERENCE_TAG}.csv"

df_wiki.to_csv(wiki_path, index=False)
df_gen_wiki.to_csv(gen_wiki_path, index=False)

print(wiki_path)
print(gen_wiki_path)

In [ ]:
wiki_auc = compute_auc_table(df_wiki)
wiki_rlb, wiki_esb = calibrate_dataset(df_wiki, df_gen_wiki)

print("AUC WikiMIA")
display(wiki_auc)

print("RLB:", wiki_rlb)
print("ESB:", wiki_esb)

In [ ]:
summary = pd.DataFrame([
    {
        "dataset": "booktection",
        "target": TARGET_TAG,
        "reference": REFERENCE_TAG,
        "method": "RLB",
        "auc": book_auc.loc[book_auc["metric"] == "run_length", "auc"].iloc[0],
        "threshold_p": book_rlb.threshold,
        "threshold_similarity": None,
        "recall": book_rlb.recall_normal,
        "fpr_adversarial": book_rlb.fpr_adversarial,
    },
    {
        "dataset": "booktection",
        "target": TARGET_TAG,
        "reference": REFERENCE_TAG,
        "method": "ESB",
        "auc": book_auc.loc[book_auc["metric"] == "esb_score", "auc"].iloc[0],
        "threshold_p": book_esb.threshold,
        "threshold_similarity": book_esb.threshold_2,
        "recall": book_esb.recall_normal,
        "fpr_adversarial": book_esb.fpr_adversarial,
    },
    {
        "dataset": f"wikimia_length{WIKIMIA_LENGTH}",
        "target": TARGET_TAG,
        "reference": REFERENCE_TAG,
        "method": "RLB",
        "auc": wiki_auc.loc[wiki_auc["metric"] == "run_length", "auc"].iloc[0],
        "threshold_p": wiki_rlb.threshold,
        "threshold_similarity": None,
        "recall": wiki_rlb.recall_normal,
        "fpr_adversarial": wiki_rlb.fpr_adversarial,
    },
    {
        "dataset": f"wikimia_length{WIKIMIA_LENGTH}",
        "target": TARGET_TAG,
        "reference": REFERENCE_TAG,
        "method": "ESB",
        "auc": wiki_auc.loc[wiki_auc["metric"] == "esb_score", "auc"].iloc[0],
        "threshold_p": wiki_esb.threshold,
        "threshold_similarity": wiki_esb.threshold_2,
        "recall": wiki_esb.recall_normal,
        "fpr_adversarial": wiki_esb.fpr_adversarial,
    },
])

summary

In [ ]:
summary_path = out_dir / f"final_summary_{TARGET_TAG}_{REFERENCE_TAG}.csv"
summary.to_csv(summary_path, index=False)

print(summary_path)